# Preprocessing Architecture

This notebook demonstrates the reusable preprocessing design for the German Credit project without fitting any transformers or transforming the dataset.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root added to path: {PROJECT_ROOT}")

Project root added to path: d:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel


## 1. Import the production preprocessing modules

In [9]:
from pathlib import Path

import pandas as pd

from src.preprocessing.feature_config import (
    TARGET_COLUMN,
    get_feature_summary,
    validate_feature_lists,
)
from src.preprocessing.preprocessing_pipeline import (
    build_scaled_preprocessor,
    build_tree_preprocessor,
)

print("Preprocessing architecture imported successfully.")

Preprocessing architecture imported successfully.


## 2. Inspect the configured feature groups

In [10]:
feature_summary = get_feature_summary()
print(feature_summary)

{'target_column': 'credit_risk', 'numerical_features': ['duration_months', 'credit_amount', 'installment_rate', 'age', 'present_residence_years', 'existing_credits', 'people_liable'], 'ordinal_features': ['checking_account_status', 'employment_duration', 'savings_account'], 'nominal_features': ['credit_history', 'purpose', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker'], 'total_predictors': 20}


## 3. Load the processed dataset schema

In [11]:
data_path = Path('../data/processed/german_credit_processed.csv')
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

validate_feature_lists(df.columns)

Dataset shape: (1000, 21)
Columns: ['checking_account_status', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings_account', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence_years', 'property', 'age', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'credit_risk']


## 4. Instantiate the reusable preprocessing builders

In [12]:
scaled_preprocessor = build_scaled_preprocessor()
tree_preprocessor = build_tree_preprocessor()

print("Scaled preprocessor type:", type(scaled_preprocessor).__name__)
print("Tree preprocessor type:", type(tree_preprocessor).__name__)
print("Scaled preprocessor steps:", [name for name, _ in scaled_preprocessor.steps])
print("Tree preprocessor steps:", [name for name, _ in tree_preprocessor.steps])

Scaled preprocessor type: Pipeline
Tree preprocessor type: Pipeline
Scaled preprocessor steps: ['preprocessor']
Tree preprocessor steps: ['preprocessor']


## 5. Feature Engineering Architecture

This section demonstrates the reusable feature engineering layer without fitting preprocessing pipelines or splitting data.

In [14]:
import importlib
import sys

module_name = "src.preprocessing.feature_engineering"
if module_name in sys.modules:
    del sys.modules[module_name]

feature_engineering = importlib.import_module(module_name)
FeatureEngineer = feature_engineering.FeatureEngineer
create_engineered_features = feature_engineering.create_engineered_features
get_engineered_feature_names = feature_engineering.get_engineered_feature_names
validate_features = feature_engineering.validate_features

engineered_df = df.copy()
original_shape = engineered_df.shape

feature_engineer = FeatureEngineer()
engineered_df = feature_engineer.fit_transform(engineered_df)

print("Original shape:", original_shape)
print("Transformed shape:", engineered_df.shape)
print("New engineered feature columns:")
print([col for col in engineered_df.columns if col in get_engineered_feature_names()])

print("\nOriginal dataframe unchanged check:")
print("credit_amount_per_month" in df.columns)
print("high_credit_amount_flag" in df.columns)
print("long_duration_flag" in df.columns)
print("age_group" in df.columns)
print("credit_duration_interaction" in df.columns)

validated_df = validate_features(engineered_df)
print("\nValidation completed successfully.")
print("Engineered feature names:", get_engineered_feature_names())

Original shape: (1000, 21)
Transformed shape: (1000, 26)
New engineered feature columns:
['credit_amount_per_month', 'high_credit_amount_flag', 'long_duration_flag', 'age_group', 'credit_duration_interaction']

Original dataframe unchanged check:
False
False
False
False
False

Validation completed successfully.
Engineered feature names: ['credit_amount_per_month', 'high_credit_amount_flag', 'long_duration_flag', 'age_group', 'credit_duration_interaction']


## 6. Notes on the architecture

This notebook intentionally stops after building the pipelines and demonstrating feature engineering. No fitting of preprocessing pipelines, no train/test split, and no model training are performed here so the design remains reusable for future training, evaluation, tuning, and deployment workflows.